In [ ]:
!pip install -q "google-genai>=2.12.1,<3" "pandas==2.2.3" "google-auth==2.49.0" pydantic

In [ ]:
MODEL_NAME = "gemini-2.0-flash"

In [ ]:

import json
from google import genai
from google.genai import types

MODEL_NAME = "gemini-2.0-flash"

API_KEY = input("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully!")

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = """
You are a healthcare appointment analyst.
Analyze patient appointment notes and identify
the main reason for missing an appointment.

Return only valid JSON matching the given schema.
Confidence must be between 0 and 1.
Use clear, concise language.
Do not invent information.
"""

In [ ]:
SYSTEM_TEXT = """
You analyse clinic appointment follow-up notes.
Return only the requested JSON.
Classify the reason for a missed appointment.
If unclear, use other.
"""

def analyse_note(note_text):

    prompt = f"""
    CONTEXT: A clinic wants to understand missed appointments.

    ROLE: You are a healthcare appointment analyst.

    ACTION: Analyse this note and identify the root cause,
    confidence score, short summary and recommended action.

    TONE: Professional, factual and concise.

    FORMAT: Return JSON with exactly four fields:
    root_cause, confidence, summary, recommended_action.

    Rules:
    - Confidence must be between 0 and 1.
    - Summary must contain at most 25 words.
    - Do not invent missing information.

    Patient note: {note_text}
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    return json.loads(response.text)

In [ ]:
result = analyse_note(
    "Patient forgot the appointment because no reminder was received."
)

print(result)

In [ ]:

{
    'root_cause': 'forgot_or_no_reminder',
    'confidence': 0.95,
    'summary': 'Patient forgot the appointment without a reminder.',
    'recommended_action': 'Re-call patient'
}

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=3)

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "other"
]

print("Pydantic model created successfully!")

In [ ]:

import pandas as pd

notes_df = pd.read_csv("followup_notes.csv")

print(notes_df.head(10))
print("Total notes:", len(notes_df))

In [ ]:

data = json.loads(response.text)
return data

In [ ]:
global LAST_RAW_RESPONSE
LAST_RAW_RESPONSE = response.text
return json.loads(LAST_RAW_RESPONSE)

In [ ]:

import time
import numpy as np
import pandas as pd
from pydantic import ValidationError

results = []
api_calls = 0

for _, row in notes_df.head(10).iterrows():

    note = row["note_text"]
    raw = ""

    record = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": np.nan,
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Check blank note first
    if pd.isna(note) or str(note).strip() == "":
        record.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank"
        })

    else:
        try:
            data = analyse_note(str(note).strip())
            api_calls += 1
            raw = LAST_RAW_RESPONSE

            record["raw_response"] = raw

            validated = FollowupInsight.model_validate(data)

            if validated.root_cause not in ALLOWED_ROOT_CAUSES:
                raise ValueError("Invalid root cause")

            record.update(validated.model_dump())
            record["validation_status"] = "valid"

        except Exception as e:
            record["validation_status"] = "invalid"
            print(f"Invalid note {row['note_id']}: {e}")

            # Preserve whatever fields the reply contained
            try:
                data
            except NameError:
                data = {}

            if isinstance(data, dict):
                for key in [
                    "root_cause", "confidence",
                    "summary", "recommended_action"
                ]:
                    if key in data:
                        record[key] = data[key]

    results.append(record)

    if record["validation_status"] != "skipped_blank":
        time.sleep(13)

notes_analysed = pd.DataFrame(results)

print("API calls:", api_calls)
print(notes_analysed[["note_id", "validation_status"]])

In [ ]:
notes_analysed.to_csv("notes_analysed.csv", index=False)

print("Saved successfully!")
print(notes_analysed.shape)

In [ ]:
from pydantic import ValidationError

bad_replies = [
    {
        "root_cause": "other",
        "confidence": 1.7,
        "summary": "Test invalid confidence",
        "recommended_action": "Re-call patient"
    },
    {
        "root_cause": "other",
        "confidence": 0.8,
        "summary": "Missing action field"
    }
]

for i, reply in enumerate(bad_replies, 1):
    try:
        FollowupInsight.model_validate(reply)
        print(f"Test {i}: Passed")
    except ValidationError as e:
        print(f"Test {i}: Validation caught")
        print(e)

In [ ]:

comparison = []

test_ids = ["N007", "N009"]

for note_id in test_ids:

    note = notes_df.loc[
        notes_df["note_id"] == note_id, "note_text"
    ].iloc[0]

    prompts = {
        "V1": f"Classify this note: {note}",
        "V2": f"""
        CONTEXT: Clinic appointment follow-up.
        ROLE: Healthcare appointment analyst.
        ACTION: Identify the root cause of the missed appointment.
        TONE: Professional and concise.
        FORMAT: Return JSON with root_cause, confidence,
        summary and recommended_action.
        Allowed root causes: {ALLOWED_ROOT_CAUSES}.
        Note: {note}
        """
    }

    for version, prompt in prompts.items():

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_TEXT,
                temperature=0,
                max_output_tokens=1000,
                response_mime_type="application/json",
                response_schema=SCHEMA,
                thinking_config=types.ThinkingConfig(
                    thinking_level="minimal"
                )
            )
        )

        try:
            result = json.loads(response.text)
            root = result["root_cause"]
            status = "Allowed" if root in ALLOWED_ROOT_CAUSES else "Invalid"
        except Exception:
            root = "Invalid JSON"
            status = "Invalid"

        comparison.append({
            "note_id": note_id,
            "prompt": version,
            "root_cause": root,
            "status": status
        })

        time.sleep(13)

comparison_df = pd.DataFrame(comparison)
print(comparison_df)

In [ ]:
print("""
V2 CRAFT prompt is recommended for deployment.
It provides clearer instructions, context, role,
action and output format.

V1 is shorter but provides less guidance.
""")

In [ ]:

appointments = pd.read_csv("appointments_clean.csv")
notes = pd.read_csv("notes_analysed.csv")

merged = pd.merge(
    notes,
    appointments,
    on="appointment_id",
    how="left"
)

print("Merged shape:", merged.shape)
display(merged.head())

In [ ]:

cross_tab = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"]
)

display(cross_tab)

In [ ]:
print("Total merged rows:", len(merged))

print("\nReminder status:")
print(merged["reminder_sent"].value_counts())

print("\nRoot cause counts:")
print(merged["root_cause"].value_counts())

In [ ]:

print("PART 4 FINDINGS")
print("-" * 40)

print("Total notes:", len(notes_analysed))
print("Valid notes:", (notes_analysed["validation_status"] == "valid").sum())
print("Invalid notes:", (notes_analysed["validation_status"] == "invalid").sum())
print("Skipped blank:", (notes_analysed["validation_status"] == "skipped_blank").sum())

print("\nRoot cause distribution:")
print(notes_analysed["root_cause"].value_counts())

print("\nReminder cross-tabulation:")
display(cross_tab)